# PTA Array NUTS Sampling

Bayesian inference over a full pulsar timing array with multiple continuous
gravitational wave (CW) sources using NumPyro NUTS (driven by JaxPINT's
`jaxpint.bayes.samplers.numpyro.run_nuts`).

**Joint parameter space:**
- Per-pulsar timing parameters (RAJ, DECJ, F0, F1, DM, PX) — whitened via WLS Cholesky
- Per-CW-source parameters (log10_h, cos_gwtheta, gwphi, log10_fgw, cos_inc, psi, phase0) — logit-transformed to unconstrained space

**Workflow:**
1. Generate N synthetic pulsars with random sky positions and timing parameters
2. Inject M CW signals into the TOAs
3. WLS-fit each pulsar individually for starting points and covariance estimates
4. Build the joint log-probability in a transformed (unconstrained) coordinate system
5. Run NUTS warmup + sampling via NumPyro
6. Diagnostics: CW parameter recovery, trace plots, corner plots

In [ ]:
# ---- User Configuration ----
N_PULSARS = 5
M_CW_SOURCES = 2
N_TOAS = 200
START_MJD = 57000.0
END_MJD = 60000.0       # ~8 yr observation span
TOA_ERROR = 1e-6         # 1 μs white noise (realistic for PTA pulsars)
FREQ = 1400.0            # MHz
SEED = 42

# NUTS configuration
N_WARMUP = 1000
N_SAMPLES = 2000

# CW parameter order within each source's GlobalParams block.  The priors
# themselves come from jaxpint.bayes.samplers.priors.cw_priors (dist.Uniform
# per param); this list only fixes the plotting/label order below.
CW_PARAM_ORDER = [
    "log10_h", "cos_gwtheta", "gwphi", "log10_fgw", "cos_inc", "psi", "phase0",
]

In [ ]:
from __future__ import annotations

import warnings
from io import StringIO

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import numpyro
import numpyro.distributions as dist

import pint.models as pm
from loguru import logger
logger.disable("pint")

from jaxpint.fitters import WLSFitter
from jaxpint.likelihood import single_pulsar_logL
from jaxpint.pta.likelihood import pta_logL
from jaxpint.types import GlobalParams
from jaxpint.pta.signals.cw import CWInjectorStack
from jaxpint.bayes.samplers.numpyro import run_nuts
from jaxpint.bayes.samplers.priors import cw_priors, resolve_priors
from jaxpint.notebook_utils import (
    generate_random_par,
    inject_and_build_config,
    pulsar_positions_from_models,
    setup_synthetic_pta,
)

print(jax.devices())

## 1. Generate Synthetic Pulsars

Each pulsar gets a random sky position, spin frequency, spindown rate, DM, and
distance. We build a minimal `.par` string and parse it with PINT.

**Convention:** `PX` is parallax in mas (PINT / `types.py` convention).
`CWInjector` converts internally to distance via `L_kpc = 1 / PX_mas`
following Ellis+2012.

In [ ]:
rng = np.random.default_rng(SEED)

par_strings = [
    generate_random_par(
        idx, rng,
        start_mjd=START_MJD,
        include_dm=True,
        free_params=True,
        extra_params={"EQUAD tel gbt": "0.1"},
    )
    for idx in range(N_PULSARS)
]
pint_models = [pm.get_model(StringIO(p)) for p in par_strings]

print(f"Generated {N_PULSARS} pulsars")
for i, m in enumerate(pint_models):
    print(f"  Pulsar {i}: {m.PSR.value}  free_params={m.free_params}")


In [ ]:
synthetic = setup_synthetic_pta(
    pint_models,
    start_mjd=START_MJD, end_mjd=END_MJD,
    n_toas=N_TOAS, toa_error_s=TOA_ERROR, freq_mhz=FREQ,
)
toa_data_list = list(synthetic.toa_data_list)
pulsar_params_list = list(synthetic.pulsar_params_list)
timing_models = list(synthetic.timing_models)
noise_models = list(synthetic.noise_models)
base_toa_data_list = list(synthetic.toa_data_list)

for i, model in enumerate(pint_models):
    px_mas = float(pulsar_params_list[i].param_value("PX"))
    distance_kpc = 1.0 / px_mas
    f0 = float(pulsar_params_list[i].param_value("F0"))
    n_free = pulsar_params_list[i].n_free
    print(
        f"  Pulsar {i}: {model.PSR.value:>20s}  "
        f"PX={px_mas:.3f} mas (d={distance_kpc:.2f} kpc)  "
        f"F0={f0:.1f} Hz  n_free={n_free}"
    )

print(f"\nAll {N_PULSARS} pulsars loaded.")

## 2. Set Up CW Sources and Inject Signals

Create M CW sources with random sky positions and GW frequencies, then inject
the CW timing delays into the TOAs. The NUTS sampler will try to recover these
injected parameters.

In [ ]:
# Pulsar ICRS unit vectors from each PINT model's RAJ/DECJ
positions = pulsar_positions_from_models(pint_models)

per_source_values = []
for m in range(M_CW_SOURCES):
    vals = {
        "log10_h": -13.0,
        "cos_gwtheta": float(rng.uniform(-1, 1)),
        "gwphi": float(rng.uniform(0, 2 * np.pi)),
        "log10_fgw": float(rng.uniform(-9, -7)),
    }
    per_source_values.append(vals)
    print(
        f"  CW source {m}: cos_gwtheta={vals['cos_gwtheta']:.3f}, "
        f"gwphi={vals['gwphi']:.3f}, log10_fgw={vals['log10_fgw']:.2f}"
    )

cw_injector = CWInjectorStack(
    positions, n_sources=M_CW_SOURCES, per_source_values=per_source_values,
)

# Register + inject signals into base TOAs; the returned `config` is the
# post-injection PTAConfig referenced later in the notebook.
true_global_params, injected_config = inject_and_build_config(synthetic, (cw_injector,))
print(f"\nGlobal params ({true_global_params.n_params}): {true_global_params.names}")


In [ ]:
# Signals already injected by inject_and_build_config; extract the resulting
# TOA data for downstream cells that expect it as a list.
injected_toa_data_list = list(injected_config.toa_data_list)
toa_data_list = injected_toa_data_list

for i, td in enumerate(injected_toa_data_list):
    delta_t = jnp.asarray(td.tdb_frac) - jnp.asarray(base_toa_data_list[i].tdb_frac)
    print(f"  Pulsar {i}: max |CW delay| = {float(jnp.max(jnp.abs(delta_t))):.2e} s")

print("\nCW signals injected into all TOAs.")


## 3. Per-Pulsar WLS Fits

Fit each pulsar individually to get starting points and covariance estimates
for the timing parameters. The WLS fit will absorb some of the slowly-varying
CW signal into the timing parameters — this is expected and the joint NUTS
sampler will disentangle them.

In [ ]:
wls_results = []
wls_params_list = []
wls_covs = []

# Fit on signal-free TOAs to get clean starting points and covariances.
# The injected CW signal would bias timing params if fitted without a CW model.
for i in range(N_PULSARS):
    fitter = WLSFitter(
        timing_models[i], toa_data_list[i], pulsar_params_list[i],
        noise_model=noise_models[i],
    )
    _ = fitter.fit_toas(maxiter=1)  # JIT warmup
    result = fitter.fit_toas(maxiter=900)

    wls_results.append(result)
    wls_params_list.append(result.params)
    wls_covs.append(np.array(result.covariance_matrix))

    print(f"  Pulsar {i}: chi2_r={result.reduced_chi2:.4f}  "
          f"free={result.params.free_names()}")

print(f"\nAll {N_PULSARS} WLS fits complete.")

## 4. Build Joint Parameter Space

The sampler operates in an unconstrained space where all parameters are O(1):

- **Timing params**: whitened via WLS Cholesky, `x_timing = mean + L @ theta_timing`
- **CW params**: logit-transformed, `x_cw = lo + (hi - lo) * sigmoid(theta_cw)`

This gives smooth gradients everywhere (no hard boundaries) and good conditioning.

In [ ]:
# inject_and_build_config already returned the post-injection PTAConfig
# (same TOAs, timing/noise models, and injectors) — reuse it directly.
config = injected_config

# Dimensions
n_cw_per_source = len(CW_PARAM_ORDER)
n_cw_total = M_CW_SOURCES * n_cw_per_source
n_free_per_pulsar = [wls_params_list[i].n_free for i in range(N_PULSARS)]
n_timing_total = sum(n_free_per_pulsar)
n_total = n_cw_total + n_timing_total
print(f"Parameter space: {n_total} dimensions")
print(f"  CW: {M_CW_SOURCES} sources x {n_cw_per_source} params = {n_cw_total}")
print(f"  Timing: {N_PULSARS} pulsars x {n_free_per_pulsar} free = {n_timing_total}")

In [ ]:
# --- Per-pulsar timing whitening (WLS Cholesky) ---
# Each pulsar: x_free = wls_mean + L @ theta_block, with theta ~ N(0, 1).
wls_means = [jnp.array(wls_params_list[i].free_values()) for i in range(N_PULSARS)]
wls_Ls = [jnp.linalg.cholesky(jnp.array(wls_covs[i])) for i in range(N_PULSARS)]

# --- CW priors: JaxPINT-native.  cw_priors() returns dist.Uniform per param;
#     NumPyro applies the unconstraining transform + Jacobian automatically, so
#     the hand-written logit/sigmoid machinery is no longer needed. ---
cw_prior_spec = cw_priors("cw0_")
for m in range(1, M_CW_SOURCES):
    cw_prior_spec = cw_prior_spec | cw_priors(f"cw{m}_")
cw_priors_resolved = resolve_priors(true_global_params.names, cw_prior_spec)

# --- Init: timing at WLS center (theta = 0); CW at each prior's midpoint. ---
init_values = {
    f"theta_timing_{i}": jnp.zeros(n_free_per_pulsar[i]) for i in range(N_PULSARS)
}
for name, d in cw_priors_resolved.items():
    init_values[name] = float((d.low + d.high) / 2.0)

print(f"CW params (native priors): {list(cw_priors_resolved)}")
print(f"Timing: {N_PULSARS} pulsars whitened via WLS Cholesky "
      f"({n_timing_total} params)")

## 5. Model & Priors

The NumPyro model combines:

- **Timing params** — a standard-normal prior on the whitened coordinates
  (`dist.Normal(0, 1)`), mapped to physical values via the WLS Cholesky factor
  `x = x_wls + L θ`. This keeps the joint geometry well-conditioned.
- **CW params** — JaxPINT-native `cw_priors()` (uniform over physical bounds);
  NumPyro unconstrains them and adds the Jacobian automatically.

Only the likelihood `pta_logL` is added as a `numpyro.factor`; both prior terms
are ordinary sample sites, so NumPyro handles all the bookkeeping.

In [ ]:
# Sanity check: evaluate the PTA likelihood + its gradient at the init point
# (WLS center for timing, CW prior midpoints).  The model below wraps this
# same likelihood with the native priors above.
gp_init = true_global_params.with_values(
    jnp.array([init_values[n] for n in true_global_params.names])
)
pp_init = tuple(
    wls_params_list[i].with_free_values(wls_means[i]) for i in range(N_PULSARS)
)


def _ll_of_gp_values(gp_values):
    gp = true_global_params.with_values(gp_values)
    return pta_logL(gp, pp_init, config)


ll0 = jax.jit(_ll_of_gp_values)(gp_init.values)
grad0 = jax.jit(jax.grad(_ll_of_gp_values))(gp_init.values)
print(f"log-likelihood at init (WLS center + CW prior midpoints): {ll0:.4f}")
print(f"Finite likelihood: {bool(jnp.isfinite(ll0))}")
print(f"Any NaN CW gradients: {bool(jnp.any(jnp.isnan(grad0)))}")

## 6. NUTS Warmup and Sampling

In [ ]:
%%time
# JaxPINT-native priors + WLS whitening, wrapped in a NumPyro model.
def model():
    # Timing: standard-normal prior on whitened coords -> physical via Cholesky.
    pulsar_params = []
    for i in range(N_PULSARS):
        theta_i = numpyro.sample(
            f"theta_timing_{i}",
            dist.Normal(0.0, 1.0).expand([n_free_per_pulsar[i]]).to_event(1),
        )
        free_vals = wls_means[i] + wls_Ls[i] @ theta_i
        pulsar_params.append(wls_params_list[i].with_free_values(free_vals))

    # CW / global params: native cw_priors() (dist.Uniform; auto-transformed).
    gp_values = jnp.stack(
        [numpyro.sample(n, cw_priors_resolved[n]) for n in true_global_params.names]
    )
    gp = true_global_params.with_values(gp_values)

    numpyro.factor("logL", pta_logL(gp, tuple(pulsar_params), config))


rng_key = jax.random.key(SEED)
print(f"Running {N_WARMUP}-step warmup + {N_SAMPLES} samples ({n_total} dimensions)...")
print("(First call includes JIT compilation — may take a few minutes)")
idata, mcmc = run_nuts(
    model,
    init=init_values,
    key=rng_key,
    num_warmup=N_WARMUP,
    num_samples=N_SAMPLES,
    max_tree_depth=10,
    target_accept_prob=0.8,
    extra_fields=("accept_prob", "num_steps", "diverging"),
    progress_bar=False,
)
extra = mcmc.get_extra_fields()

In [ ]:
%%time
samples = mcmc.get_samples()
acceptance_rate = float(jnp.mean(extra["accept_prob"]))
print(f"Acceptance rate: {acceptance_rate:.2%}")
print(f"Sampled sites: {list(samples)}")

## 7. Diagnostics

In [ ]:
# Transform samples to physical space.
samples = mcmc.get_samples()

# CW: sampled directly in physical space by the native priors.
cw_samples_np = np.stack(
    [np.asarray(samples[n]) for n in true_global_params.names], axis=1
)  # (N_SAMPLES, n_cw_total)

# Per-pulsar timing: whitened -> physical via the Cholesky factor.
timing_samples_per_pulsar = []
for i in range(N_PULSARS):
    theta_blocks = samples[f"theta_timing_{i}"]              # (N_SAMPLES, n_f)
    physical = jax.vmap(
        lambda tb, m=wls_means[i], L=wls_Ls[i]: m + L @ tb
    )(theta_blocks)
    timing_samples_per_pulsar.append(np.array(physical))

print(f"CW samples: {cw_samples_np.shape}")
print(f"Timing samples per pulsar: {[s.shape for s in timing_samples_per_pulsar]}")

In [ ]:
# NUTS efficiency metrics
num_steps = np.array(extra["num_steps"])
print(f"Acceptance rate: {acceptance_rate:.2%}")
print(f"Mean integration steps: {num_steps.mean():.1f}")
print(f"Max integration steps: {int(num_steps.max())}")
print(f"Divergent transitions: {int(np.array(extra['diverging']).sum())}")

### CW Parameter Trace Plots

Each row shows one CW parameter across all sources. Red dashed lines mark the
injected true values.

In [ ]:
true_cw_np = np.array(true_global_params.values)

fig, axes = plt.subplots(n_cw_total, 2, figsize=(14, 2.5 * n_cw_total))
if n_cw_total == 1:
    axes = axes[np.newaxis, :]

for k in range(n_cw_total):
    # Determine source index and param name
    src_idx = k // n_cw_per_source
    param_idx = k % n_cw_per_source
    param_name = f"cw{src_idx}_{CW_PARAM_ORDER[param_idx]}"

    # Trace plot
    axes[k, 0].plot(cw_samples_np[:, k], alpha=0.7, linewidth=0.5)
    axes[k, 0].axhline(true_cw_np[k], color="r", linestyle="--", label="True")
    axes[k, 0].set_ylabel(param_name, fontsize=8)
    axes[k, 0].legend(loc="upper right", fontsize=7)

    # Histogram
    axes[k, 1].hist(cw_samples_np[:, k], bins=50, density=True, alpha=0.7)
    axes[k, 1].axvline(true_cw_np[k], color="r", linestyle="--", label="True")
    axes[k, 1].legend(loc="upper right", fontsize=7)

axes[0, 0].set_title("Trace")
axes[0, 1].set_title("Marginal posterior")
axes[-1, 0].set_xlabel("Sample")
axes[-1, 1].set_xlabel("Value")
plt.tight_layout()
plt.show()

### CW Corner Plots

Pairwise scatter plots for each CW source's 7 parameters.

In [ ]:
for src in range(M_CW_SOURCES):
    src_start = src * n_cw_per_source
    src_end = src_start + n_cw_per_source
    src_samples = cw_samples_np[:, src_start:src_end]
    src_true = true_cw_np[src_start:src_end]
    labels = CW_PARAM_ORDER

    fig, axes = plt.subplots(n_cw_per_source, n_cw_per_source,
                             figsize=(14, 14))
    for i in range(n_cw_per_source):
        for j in range(n_cw_per_source):
            ax = axes[i, j]
            if j > i:
                ax.set_visible(False)
            elif i == j:
                ax.hist(src_samples[:, i], bins=40, density=True, alpha=0.7)
                ax.axvline(src_true[i], color="r", linestyle="--")
            else:
                ax.scatter(src_samples[:, j], src_samples[:, i],
                           s=1, alpha=0.3)
                ax.axhline(src_true[i], color="r", linestyle="--", lw=0.5)
                ax.axvline(src_true[j], color="r", linestyle="--", lw=0.5)

            if i == n_cw_per_source - 1:
                ax.set_xlabel(labels[j], fontsize=7)
            if j == 0:
                ax.set_ylabel(labels[i], fontsize=7)
            ax.tick_params(labelsize=5)

    fig.suptitle(f"CW source {src}", fontsize=14)
    plt.tight_layout()
    plt.show()

### CW Parameter Recovery Summary

In [ ]:
print(f"{'Parameter':<25} {'True':>12} {'Post Mean':>12} {'Post Std':>12} {'Diff (σ)':>10}")
print("-" * 72)

for src in range(M_CW_SOURCES):
    for k, pname in enumerate(CW_PARAM_ORDER):
        idx = src * n_cw_per_source + k
        true_val = true_cw_np[idx]
        post_mean = np.mean(cw_samples_np[:, idx])
        post_std = np.std(cw_samples_np[:, idx])
        diff_sigma = (post_mean - true_val) / post_std if post_std > 0 else 0.0
        label = f"cw{src}_{pname}"
        print(f"{label:<25} {true_val:>12.4f} {post_mean:>12.4f} {post_std:>12.4g} {diff_sigma:>10.2f}")
    print()

### Per-Pulsar Timing Parameter Recovery

In [ ]:
for i in range(N_PULSARS):
    samples_i = timing_samples_per_pulsar[i]
    free_names = wls_params_list[i].free_names()
    true_free = np.array(pulsar_params_list[i].free_values())
    wls_free = np.array(wls_params_list[i].free_values())
    wls_unc = np.array(wls_results[i].parameter_uncertainties)

    print(f"\nPulsar {i}: {pint_models[i].PSR.value}")
    print(f"  {'Param':<10} {'True':>18} {'WLS':>18} {'Post Mean':>18} {'Post Std':>12} {'Δ(σ)':>8}")
    print("  " + "-" * 80)
    for j, name in enumerate(free_names):
        post_mean = np.mean(samples_i[:, j])
        post_std = np.std(samples_i[:, j])
        diff = (post_mean - true_free[j]) / post_std if post_std > 0 else 0.0
        print(f"  {name:<10} {true_free[j]:>18.10g} {wls_free[j]:>18.10g} "
              f"{post_mean:>18.10g} {post_std:>12.4g} {diff:>8.2f}")

### Effective Sample Size

ESS estimates the number of independent samples by accounting for autocorrelation.
Parameters with ESS < 100 may need longer chains.

In [ ]:
def ess_bulk(chain):
    """Estimate bulk ESS for a 1-D chain using initial positive sequence estimator."""
    n = len(chain)
    chain = chain - np.mean(chain)
    # Compute autocorrelation via FFT
    fft_chain = np.fft.fft(chain, n=2 * n)
    acf = np.fft.ifft(fft_chain * np.conj(fft_chain))[:n].real
    acf /= acf[0]
    # Sum consecutive pairs until negative
    tau = 1.0
    for t in range(1, n // 2):
        pair_sum = acf[2 * t - 1] + acf[2 * t]
        if pair_sum < 0:
            break
        tau += 2.0 * pair_sum
    return n / tau


# Compute ESS for all parameters
all_theta_samples = np.concatenate(
    [cw_samples_np, *timing_samples_per_pulsar], axis=1
)
all_names = []
for src in range(M_CW_SOURCES):
    for pname in CW_PARAM_ORDER:
        all_names.append(f"cw{src}_{pname}")
for i in range(N_PULSARS):
    for name in wls_params_list[i].free_names():
        all_names.append(f"psr{i}_{name}")

ess_values = [ess_bulk(all_theta_samples[:, k]) for k in range(n_total)]

print(f"{'Parameter':<25} {'ESS':>8}")
print("-" * 35)
for name, ess in zip(all_names, ess_values):
    flag = " <<<" if ess < 100 else ""
    print(f"{name:<25} {ess:>8.0f}{flag}")

print(f"\nMin ESS: {min(ess_values):.0f}")
print(f"Median ESS: {np.median(ess_values):.0f}")